# Analyse des mouvements de mains au piano — version MediaPipe

Pipeline : **vidéo → détection et suivi des 2 mains (21 points chacune) → nettoyage et lissage → trajectoires, vitesses, rapport de qualité → vidéo annotée + export CSV.**

### Ce qui change par rapport à la version MMPose
- **Une seule dépendance principale** (`mediapipe`) : plus de patch de `site-packages`, plus de mmcv/mmdet, plus de recadrage manuel ni de PNG intermédiaires.
- **Suivi temporel** : MediaPipe est utilisé en mode `VIDEO` (il suit les mains d'une frame à l'autre au lieu de les redétecter).
- **Identité gauche/droite robuste** : appariement par proximité du poignet d'une frame à l'autre (fonctionne aussi quand les mains se croisent), au lieu d'un tri sur la position en x.
- **Axe du temps corrigé** : le temps vient des vrais numéros de frame et du fps lu dans la vidéo (avant : `step/59`, avec un `step` qui ne valait pas le pas d'extraction, et des doublons supprimés qui décalaient le temps).
- **Frames sans détection gérées** (avant : erreur si une seule main était trouvée).
- **Nettoyage** : rejet des valeurs aberrantes, interpolation des petits trous, lissage Savitzky-Golay.
- **Vitesses normalisées par la taille de la main** : comparables d'une vidéo à l'autre, indépendantes de la distance à la caméra.
- **Rapport de qualité chiffré** : taux de détection, jitter avant/après lissage.

## 0. Installation et configuration

In [ ]:
# Fonctionne dans Colab comme en local (Python 3.9 à 3.12).
# OpenCV (cv2) est installé automatiquement avec mediapipe.
%pip install -q mediapipe numpy pandas matplotlib scipy pillow

In [ ]:
from pathlib import Path
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from scipy.optimize import linear_sum_assignment
from scipy.signal import savgol_filter
from PIL import Image as PILImage
from IPython.display import Image as IPImage, display

# --- Constantes de la main (topologie MediaPipe = 21 points, identique à COCO-WholeBody) ---
HAND_SKELETON = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # pouce
    (0, 5), (5, 6), (6, 7), (7, 8),        # index
    (0, 9), (9, 10), (10, 11), (11, 12),   # majeur
    (0, 13), (13, 14), (14, 15), (15, 16), # annulaire
    (0, 17), (17, 18), (18, 19), (19, 20), # auriculaire
]
FINGERTIPS = [4, 8, 12, 16, 20]
FINGER_NAMES = ["pouce", "index", "majeur", "annulaire", "auriculaire"]
HAND_NAMES = ["main gauche", "main droite"]       # indices 0 et 1 partout dans le notebook
HAND_COLORS_BGR = [(255, 140, 30), (30, 140, 255)]  # gauche : bleu, droite : orange (BGR pour OpenCV)

# --- Chemins ---
DATA_DIR = Path("data")
OUT_DIR = Path("output")
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

VIDEO_PATH = DATA_DIR / "gamme_do_majeur.mp4"       # <- votre vidéo (vue de dessus)
IMAGE_PATH = DATA_DIR / "deux_mains.jpg"            # <- optionnel : une image pour tester
MODEL_PATH = DATA_DIR / "hand_landmarker.task"
MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
             "hand_landmarker/float16/1/hand_landmarker.task")

# --- Paramètres ---
STRIDE = 1          # 1 = toutes les frames (recommandé : un doigt bouge vite), 2 = une sur deux...
MIN_CONF = 0.5      # seuil de confiance MediaPipe (détection, présence, suivi)
MAX_GAP_S = 0.15    # on interpole les trous de détection jusqu'à cette durée (secondes)
SMOOTH_S = 0.12     # fenêtre de lissage (secondes)
POLY = 2            # degré du polynôme Savitzky-Golay
OUTLIER_HS = 0.4    # un point qui s'écarte de plus de 0,4 x taille de la main de sa médiane locale est rejeté

In [ ]:
# Téléchargement du modèle de détection de mains (une seule fois)
if not MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Modèle prêt :", MODEL_PATH, f"({MODEL_PATH.stat().st_size / 1e6:.1f} Mo)")

# Vidéo : dans Colab, on propose l'import si le fichier est absent
if not VIDEO_PATH.exists():
    try:
        from google.colab import files
        print("Importez votre vidéo :")
        uploaded = files.upload()
        name = next(iter(uploaded))
        Path(name).rename(VIDEO_PATH)
    except ImportError:
        print(f"Placez votre vidéo dans {VIDEO_PATH}")

## 1. Détection et suivi des mains

Trois fonctions utilitaires :
- `make_landmarker` : crée le détecteur MediaPipe (mode `VIDEO` avec suivi, ou mode `IMAGE` pour une photo seule).
- `drop_duplicates` : MediaPipe détecte parfois deux fois la même main (une « gauche » et une « droite » superposées) ; on garde la plus confiante.
- `assign_hands` : attribue chaque détection à la main gauche (0) ou droite (1) du pianiste en suivant les poignets d'une frame à l'autre.

> **Pourquoi ne pas utiliser le label « Left/Right » de MediaPipe ?** Il suppose une image en miroir (selfie) et se trompe facilement en vue de dessus, dos de la main face à la caméra. Le suivi du poignet est plus fiable ici. Convention : *main gauche = main gauche du pianiste*, située à gauche de l'image au départ.

In [ ]:
def make_landmarker(video_mode=True, num_hands=2, conf=MIN_CONF):
    options = vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=vision.RunningMode.VIDEO if video_mode else vision.RunningMode.IMAGE,
        num_hands=num_hands,
        min_hand_detection_confidence=conf,
        min_hand_presence_confidence=conf,
        min_tracking_confidence=conf,
    )
    return vision.HandLandmarker.create_from_options(options)


def result_to_detections(result, width, height):
    """Résultat MediaPipe -> liste de (points (21,2) en pixels, score)."""
    detections = []
    for landmarks, handedness in zip(result.hand_landmarks, result.handedness):
        pts = np.array([[p.x * width, p.y * height] for p in landmarks], dtype=np.float32)
        detections.append((pts, handedness[0].score))
    return detections


def drop_duplicates(detections):
    """Si deux détections ont quasiment le même poignet, on garde la plus confiante."""
    if len(detections) == 2:
        (a, score_a), (b, score_b) = detections
        hand_size = np.linalg.norm(a[9] - a[0]) + 1e-6
        if np.linalg.norm(a[0] - b[0]) < 0.5 * hand_size:
            return [detections[0] if score_a >= score_b else detections[1]]
    return detections


def assign_hands(detections, last_wrist, default_x):
    """Associe les détections aux mains 0 (gauche) et 1 (droite).

    last_wrist : dernière position connue du poignet de chaque main (ou None)
    default_x  : position x supposée de chaque main tant qu'on ne l'a jamais vue
    """
    out = {0: None, 1: None}
    if not detections:
        return out
    wrists = np.array([pts[0] for pts, _ in detections])
    cost = np.zeros((len(detections), 2))
    for i, w in enumerate(wrists):
        for h in (0, 1):
            if last_wrist[h] is not None:
                cost[i, h] = np.linalg.norm(w - last_wrist[h])
            else:
                cost[i, h] = abs(w[0] - default_x[h])
    rows, cols = linear_sum_assignment(cost)   # appariement optimal (algorithme hongrois)
    for r, c in zip(rows, cols):
        out[c] = detections[r]
    return out


def draw_hands(frame_bgr, hands, scale=1):
    """Dessine les squelettes. hands : (2, 21, 2), les NaN sont ignorés."""
    for h, pts in enumerate(hands):
        color = HAND_COLORS_BGR[h]
        ok = np.isfinite(pts).all(axis=1)
        for i, j in HAND_SKELETON:
            if ok[i] and ok[j]:
                p, q = (int(round(v)) for v in pts[i]), (int(round(v)) for v in pts[j])
                cv2.line(frame_bgr, tuple(p), tuple(q), color, 2 * scale, cv2.LINE_AA)
        for k in range(21):
            if ok[k]:
                tip = k in FINGERTIPS
                cv2.circle(frame_bgr, (int(round(pts[k, 0])), int(round(pts[k, 1]))),
                           (5 if tip else 3) * scale, (255, 255, 255) if tip else color, -1, cv2.LINE_AA)
    return frame_bgr

### Test rapide sur une image (optionnel)
Même détecteur en mode `IMAGE`. Il n'y a plus de problème de « bounding box = image entière » comme avec MMPose : MediaPipe trouve lui-même les mains dans l'image.

In [ ]:
def detect_on_image(path):
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise FileNotFoundError(path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    height, width = rgb.shape[:2]
    with make_landmarker(video_mode=False) as detector:
        result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb)))
    detections = drop_duplicates(result_to_detections(result, width, height))
    assigned = assign_hands(detections, [None, None], [0.25 * width, 0.75 * width])

    hands = np.full((2, 21, 2), np.nan, dtype=np.float32)
    for h, det in assigned.items():
        if det is not None:
            hands[h] = det[0]
    annotated = draw_hands(bgr.copy(), hands, scale=max(1, width // 800))

    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"{len(detections)} main(s) détectée(s)")
    plt.show()
    return hands


if IMAGE_PATH.exists():
    _ = detect_on_image(IMAGE_PATH)
else:
    print(f"(pas d'image de test : {IMAGE_PATH})")

## 2. Extraction des keypoints sur la vidéo

On lit la vidéo une seule fois. Les timestamps passés à MediaPipe viennent du numéro de frame et du fps réel de la vidéo (le mode `VIDEO` exige des timestamps croissants).

Résultat : un tableau `kp_raw` de forme `(frames, 2 mains, 21 points, 2 coordonnées x/y)` en pixels, avec `NaN` là où la main n'est pas détectée.

In [ ]:
def extract_keypoints(video_path, stride=1):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Impossible d'ouvrir {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    kp_list, score_list, frame_ids = [], [], []
    last_wrist = [None, None]
    default_x = [0.25 * width, 0.75 * width]

    idx = 0
    with make_landmarker(video_mode=True) as detector:
        while cap.grab():
            if idx % stride == 0:
                ok, frame = cap.retrieve()
                if not ok:
                    break
                rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                timestamp_ms = int(round(idx * 1000 / fps))
                result = detector.detect_for_video(
                    mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), timestamp_ms)

                detections = drop_duplicates(result_to_detections(result, width, height))
                assigned = assign_hands(detections, last_wrist, default_x)

                kp = np.full((2, 21, 2), np.nan, dtype=np.float32)
                score = np.full(2, np.nan, dtype=np.float32)
                for h, det in assigned.items():
                    if det is not None:
                        kp[h], score[h] = det
                        last_wrist[h] = det[0][0]
                kp_list.append(kp)
                score_list.append(score)
                frame_ids.append(idx)
                if len(frame_ids) % 200 == 0:
                    print(f"  {idx}/{total} frames traitées")
            idx += 1
    cap.release()
    return (np.stack(kp_list), np.stack(score_list), np.array(frame_ids), fps, (width, height))


kp_raw, scores, frame_ids, fps, (W, H) = extract_keypoints(VIDEO_PATH, STRIDE)
fps_eff = fps / STRIDE                  # cadence effective des frames traitées
times = frame_ids / fps                 # temps exact de chaque frame traitée (secondes)
print(f"{len(frame_ids)} frames traitées | vidéo {W}x{H} à {fps:.2f} fps | durée {times[-1]:.1f} s")

## 3. Nettoyage et lissage

Dans l'ordre :
1. **Taille de la main** (distance poignet → base du majeur, médiane sur la vidéo) : sert d'unité pour tout normaliser.
2. **Rejet des aberrations** : un point trop éloigné de la médiane glissante de sa propre trajectoire est supprimé.
3. **Interpolation** des petits trous (≤ `MAX_GAP_S`) ; les longs trous restent `NaN` plutôt que d'inventer des données.
4. **Lissage** Savitzky-Golay segment par segment (sans jamais lisser à travers un trou).

In [ ]:
def hand_sizes(kp):
    """Taille de chaque main (px) = distance poignet -> base du majeur, médiane sur la vidéo."""
    d = np.linalg.norm(kp[:, :, 9, :] - kp[:, :, 0, :], axis=-1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmedian(d, axis=0)


def reject_outliers(kp, sizes, win=5, thr_hs=OUTLIER_HS):
    out = kp.copy()
    for h in range(2):
        if np.isnan(sizes[h]):
            continue
        for k in range(21):
            xy = pd.DataFrame(kp[:, h, k, :])
            med = xy.rolling(win, center=True, min_periods=1).median().to_numpy()
            dist = np.linalg.norm(xy.to_numpy() - med, axis=1)
            out[dist > thr_hs * sizes[h], h, k, :] = np.nan
    return out


def interpolate_gaps(a, max_gap):
    """Interpole linéairement les trous de NaN de longueur <= max_gap (pas aux extrémités)."""
    out = a.copy()
    isnan = np.isnan(a)
    if not isnan.any() or isnan.all():
        return out
    idx = np.arange(len(a))
    filled = np.interp(idx, idx[~isnan], a[~isnan])
    i, n = 0, len(a)
    while i < n:
        if isnan[i]:
            j = i
            while j < n and isnan[j]:
                j += 1
            if i > 0 and j < n and (j - i) <= max_gap:
                out[i:j] = filled[i:j]
            i = j
        else:
            i += 1
    return out


def smooth_series(a, window, poly):
    """Savitzky-Golay appliqué séparément sur chaque segment sans NaN."""
    out = np.full_like(a, np.nan)
    valid = ~np.isnan(a)
    i, n = 0, len(a)
    while i < n:
        if valid[i]:
            j = i
            while j < n and valid[j]:
                j += 1
            seg = a[i:j]
            w = min(window, len(seg) if len(seg) % 2 == 1 else len(seg) - 1)   # fenêtre impaire <= segment
            out[i:j] = savgol_filter(seg, w, poly) if w > poly else seg
            i = j
        else:
            i += 1
    return out


def clean_and_smooth(kp, sizes, fps_eff):
    max_gap = max(1, int(round(MAX_GAP_S * fps_eff)))
    window = max(int(round(SMOOTH_S * fps_eff)), POLY + 2)
    window += (window % 2 == 0)

    kp_clean = reject_outliers(kp, sizes)
    kp_filled = kp_clean.astype(np.float64)
    kp_smooth = np.full_like(kp_filled, np.nan)
    for h in range(2):
        for k in range(21):
            for c in range(2):
                s = interpolate_gaps(kp_filled[:, h, k, c], max_gap)
                kp_filled[:, h, k, c] = s
                kp_smooth[:, h, k, c] = smooth_series(s, window, POLY)
    return kp_filled, kp_smooth, window, max_gap


sizes = hand_sizes(kp_raw)
kp_filled, kp_smooth, window, max_gap = clean_and_smooth(kp_raw, sizes, fps_eff)
print(f"Taille de main (px) : gauche = {sizes[0]:.0f}, droite = {sizes[1]:.0f}")
print(f"Lissage : fenêtre de {window} frames ({window / fps_eff * 1000:.0f} ms) | trous comblés jusqu'à {max_gap} frames")

## 4. Rapport de qualité

Des chiffres plutôt qu'une impression visuelle :
- **Taux de détection** : part des frames où la main est trouvée (brut), puis après interpolation.
- **Jitter** : accélération moyenne des points (en tailles de main / s²). Un point qui « tremble » a une accélération élevée ; le lissage doit la faire baisser sans effacer les vrais mouvements.

In [ ]:
def jitter(kp, sizes, fps_eff):
    acc = np.diff(kp, n=2, axis=0) * fps_eff ** 2          # px / s²
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(np.linalg.norm(acc, axis=-1), axis=(0, 2)) / sizes


def detection_rate(kp):
    return 100 * np.mean(~np.isnan(kp[:, :, 0, 0]), axis=0)


with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    mean_scores = np.nanmean(scores, axis=0)

report = pd.DataFrame({
    "main": HAND_NAMES,
    "détection brute (%)": detection_rate(kp_raw).round(1),
    "après interpolation (%)": detection_rate(kp_smooth).round(1),
    "score MediaPipe moyen": mean_scores.round(3),
    "jitter brut": jitter(kp_raw, sizes, fps_eff).round(1),
    "jitter lissé": jitter(kp_smooth, sizes, fps_eff).round(1),
}).set_index("main")
report

## 5. Trajectoires et vitesses des bouts de doigts

Les vitesses sont calculées avec le **vrai temps** de chaque frame et exprimées en **tailles de main par seconde**.

In [ ]:
def compute_speeds(kp_s, times, sizes):
    velocity = np.gradient(kp_s, times, axis=0)          # px / s, supporte un pas de temps irrégulier
    return np.linalg.norm(velocity, axis=-1) / sizes[None, :, None]   # (frames, 2, 21)


speeds = compute_speeds(kp_smooth, times, sizes)

# --- Trajectoires des bouts de doigts, superposées à la première image ---
cap = cv2.VideoCapture(str(VIDEO_PATH))
_, first_frame = cap.read()
cap.release()
first_rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
finger_colors = plt.cm.tab10(np.arange(5))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for h, ax in enumerate(axes):
    ax.imshow(first_rgb)
    for f, tip in enumerate(FINGERTIPS):
        ax.plot(kp_smooth[:, h, tip, 0], kp_smooth[:, h, tip, 1], color=finger_colors[f], lw=1.5, label=FINGER_NAMES[f])
    pts = kp_smooth[:, h].reshape(-1, 2)
    if np.isfinite(pts).any():
        (x0, y0), (x1, y1) = np.nanmin(pts, axis=0) - 60, np.nanmax(pts, axis=0) + 60
        ax.set_xlim(max(x0, 0), min(x1, W))
        ax.set_ylim(min(y1, H), max(y0, 0))
    ax.set_title(f"Trajectoires des bouts de doigts — {HAND_NAMES[h]}")
    ax.axis("off")
axes[0].legend(loc="lower left")
plt.tight_layout()
plt.show()

# --- Vitesses des bouts de doigts au cours du temps ---
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for h, ax in enumerate(axes):
    for f, tip in enumerate(FINGERTIPS):
        ax.plot(times, speeds[:, h, tip], color=finger_colors[f], lw=1.2, label=FINGER_NAMES[f])
    ax.set_ylabel("vitesse (taille de main / s)")
    ax.set_title(HAND_NAMES[h])
    ax.grid(alpha=0.3)
axes[0].legend(ncol=5, loc="upper right")
axes[1].set_xlabel("temps (s)")
plt.tight_layout()
plt.show()

**Comment lire ces courbes** : un pic de vitesse sur un doigt correspond à un déplacement de ce doigt (changement de position, passage du pouce, frappe). Avec la gamme de do majeur, on s'attend à voir le pouce accélérer légèrement avant les autres au passage du pouce sous la main (mi → fa en montée), et le majeur en descente. À vous de vérifier si c'est le cas avec les nouveaux calculs.

## 6. Vidéo annotée et GIF

On exporte une vidéo MP4 (à télécharger) et un GIF allégé (pour l'affichage dans le notebook et le README).

In [ ]:
def render_outputs(video_path, kp_s, frame_ids, fps_eff, out_mp4, out_gif, gif_width=480, gif_max_frames=150):
    cap = cv2.VideoCapture(str(video_path))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    scale = max(1, width // 800)

    writer = cv2.VideoWriter(str(out_mp4), cv2.VideoWriter_fourcc(*"mp4v"), fps_eff, (width, height))
    lookup = {int(fid): k for k, fid in enumerate(frame_ids)}
    gif_every = max(1, int(np.ceil(len(frame_ids) / gif_max_frames)))
    gif_frames = []

    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        k = lookup.get(idx)
        if k is not None:
            draw_hands(frame, kp_s[k], scale=scale)
            writer.write(frame)
            if k % gif_every == 0:
                small = cv2.resize(frame, (gif_width, int(height * gif_width / width)), interpolation=cv2.INTER_AREA)
                gif_frames.append(PILImage.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB)))
        idx += 1
    writer.release()
    cap.release()

    duration_ms = max(20, int(1000 / fps_eff * gif_every))
    gif_frames[0].save(out_gif, save_all=True, append_images=gif_frames[1:], duration=duration_ms, loop=0)


OUT_MP4 = OUT_DIR / "mains_annotees.mp4"
OUT_GIF = OUT_DIR / "mains_annotees.gif"
render_outputs(VIDEO_PATH, kp_smooth, frame_ids, fps_eff, OUT_MP4, OUT_GIF)
print("Écrit :", OUT_MP4, "et", OUT_GIF)
display(IPImage(filename=str(OUT_GIF)))

## 7. Export des données

Un CSV « tidy » (une ligne par frame × main × point) : directement exploitable avec pandas, Excel ou pour entraîner un modèle plus tard.

In [ ]:
n = len(frame_ids)
fi, hi, ki = np.meshgrid(np.arange(n), np.arange(2), np.arange(21), indexing="ij")
fi, hi, ki = fi.ravel(), hi.ravel(), ki.ravel()

df = pd.DataFrame({
    "frame": frame_ids[fi],
    "time_s": times[fi],
    "hand": np.array(["gauche", "droite"])[hi],
    "keypoint": ki,
    "x_raw": kp_raw[..., 0].ravel(),
    "y_raw": kp_raw[..., 1].ravel(),
    "x": kp_smooth[..., 0].ravel(),
    "y": kp_smooth[..., 1].ravel(),
    "speed_hand_sizes_per_s": speeds.ravel(),
})
df.to_csv(OUT_DIR / "keypoints.csv", index=False)
print(f"{len(df)} lignes écrites dans {OUT_DIR / 'keypoints.csv'}")
df.head()

## 8. Pistes pour la suite

1. **Détection du clavier** (coins + homographie) pour passer des pixels aux touches, puis associer chaque bout de doigt à une touche.
2. **Détection des frappes** : en vue de dessus la profondeur est ambiguë ; une 2ᵉ caméra de côté, ou l'audio (`librosa` / `basic-pitch`) ou un export MIDI comme vérité terrain, permettent d'aligner « note jouée » et « doigt ».
3. **Fine-tuning** d'un modèle de pose (ex. RTMPose) sur 200-500 frames annotées de vos vidéos, puis comparaison chiffrée avec MediaPipe (PCK, jitter).
4. **Filtre One Euro** pour un lissage en temps réel (webcam).
5. **3D** : `result.hand_world_landmarks` donne des coordonnées pseudo-3D par main ; deux caméras permettent une vraie triangulation.
6. **Comparaison élève / référence** avec du DTW sur les courbes de vitesse.